In [1]:
import numpy as np
import pandas as pd
import datetime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
import duckdb

In [2]:
df = pd.read_excel('urbox_bi_analyst_test.xlsx', sheet_name = 'Data for Part 1')

In [21]:
k= df.groupby(['user_id'], as_index = False).agg({'brand_id' : 'nunique', 'transaction_id': 'count'})

In [22]:
k.describe()

,user_id,brand_id,transaction_id
count,1.000000e+03,1000.000000,1000.000000
mean,9.624309e+08,1.780000,5.652000
std,2.049425e+08,1.327917,7.359964
min,1.084650e+05,1.000000,1.000000
25%,1.002285e+09,1.000000,1.000000
50%,1.005895e+09,1.000000,3.000000
75%,1.009344e+09,2.000000,7.000000
max,1.016405e+09,10.000000,50.000000


Số brand trung bình 1 user là 1.78, khoảng 75% user chỉ quy đổi sang 2 brand
Số lượt giao dịch trung bình 1 user là 5.65, khoảng 75% user chỉ dừng ở 7 giao dịch
Nhiệm vụ là cần tăng số brand sử dụng và số lần giao dịch lên. 

Dựa theo bảng cross-sales đã được thực hiện ở câu hỏi trước và tần suất sử dụng các brand theo thời gian cho thấy các đặc điểm có thể khai thác được để tăng số brand user sử dụng và số lần giao dịch.

Trước khi thực hiện các giải pháp cần dịch chuyển người dùng sử dụng app của Urbox để quy đổi e-voucher, dữ liệu hành vi khách hàng sẽ được ghi lại để đề xuất các gói giải pháp sau:

Bài toán 1: Tăng số lượng brand trung bình 1 khách hàng sử dụng:
1.1 Giải pháp 1 : Đối với bảng cross-sales đã thực hiện trước đó, hệ thống sẽ tự động đề xuất cho khách hàng brand tiếp theo khi khách đã quy đổi một brand, ưu tiên những gói cross-sales có lượng user quy đổi lớn, kế tiếp tới giới thiệu các combo khác theo thứ tự.
1.2 Giải pháp 2: Đưa ra chính sách thưởng nếu khách sử dụng voucher cho các mốc 3-5-10,.. Brand khác nhau trong một khoảng thời gian nhất định và sau thời điểm đó thì hệ thống sẽ đếm lại từ đầu.
Phần thưởng có thể quy đổi như nếu khách đổi khi chạm mốc 3 brand sẽ được +3 điểm, nhưng chạm 5 brand sẽ được +10đ, chạm mốc 10 brand sẽ được +15 điểm. Điểm có thể quy đổi sang tiền e-voucher tương ứng.
1.3 Giải pháp 3: Đối với các brand mới mà khách chưa từng có lịch sử quy đổi voucher nhưng có lượt sử dụng nhiều ở khách khác thì hệ thống sẽ đề xuất cho khách hàng

Bài toán 2: Tăng số lượng giao dịch trung bình trên 1 khách hàng
2.1 Giải pháp 1: App sẽ ghi nhận các chương trình khuyến mãi của từng brand và đề xuất cho khách hàng nếu brand đó khách hàng đã có lịch sử sử dụng.
2.2 Giải pháp 2: Một số brand có lượt sử dụng lớn như brand 1511, 82, 395, 552 được ưu tiên xuất hiện ở vị trí dễ thấy nhất trong app. 
Một số brand thường được sử dụng ngay khi khách có voucher như 1511 sẽ được gửi đề xuất cho khách hàng khi user_id của khách được kích hoạt (giá trị e-voucher > 0).
2.3 Giải pháp 3: Dựa theo tích chất của từng brand để đưa đề xuất đúng thời điểm, ví dụ như voucher siêu thị, khu vui chơi cho trẻ em thì sẽ đề xuất vào cuối tuần, voucher cà phê, bánh mì được đẩy vào đầu tuần làm việc.
2.4 Giải pháp 4: Tích điểm cho các mốc số lượng giao dịch đã được thực hiện trên App của Urbox, cần tăng số lượng lên trên mức 6 giao dịch với mỗi user, ví dụ như mốc 10-20-30 lần (có giới hạn nhỏ nhất cho mỗi lần quy đổi, ví dụ 100k).
2.5 Giải pháp 5: Nếu như tài khoản của khách chỉ còn 1 số tiền lẻ như 10k, 20k, app sẽ đề xuất các brand có sản phẩm trong tầm hạn mức đó cho khách hàng để giúp khách hàng tránh lãng phí trước khi e-voucher hết hạn
